In [3]:
import torch
import torch.nn.functional as F
import torchaudio
from torch.utils.data import DataLoader
import auraloss

from torch.utils.tensorboard import SummaryWriter

from preprocess import createDataset
from tqdm import tqdm

from models import Mamba
from config import Hyperparams, ModelParams

args = ModelParams()
h_params = Hyperparams()

if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

print("using", device)

using cuda


In [4]:
train_data, val_data, test_data = createDataset(h_params)
train_dataloader = DataLoader(train_data, batch_size=h_params.batch_size, shuffle=True)
val_dataloader = DataLoader(val_data, batch_size=h_params.batch_size, shuffle=True)

len(train_dataloader)

100%|██████████| 25/25 [00:01<00:00, 13.78it/s]


165

In [5]:
class Loss:
    def __init__(self):
        self.loss_fn1 = torch.nn.MSELoss()
        self.loss_fn2 = auraloss.freq.MultiResolutionSTFTLoss(fft_sizes = [1024, 512, 256], hop_sizes = [120, 50, 25], win_lengths = [440, 240, 100], mag_distance = "L1")

    def compute(self, output, target):
        """
        Input:
            target (float32):   (B, L, 1)
            output (float32):   (B, L, 1)
        Return:
            loss (float32): (1)
        """
        return self.loss_fn1(output, target) + h_params.alpha * self.loss_fn2(output.permute(0,2,1), target.permute(0,2,1))

In [6]:
model = Mamba(args).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr = 0.001)
loss_fn = Loss()
print(sum(p.numel() for p in model.parameters()), "model parameters\n")
print("Model architecture:")
print(model)

31504 model parameters

Model architecture:
Mamba(
  (film_gen): FiLMGenerator(
    (fc): Sequential(
      (0): Linear(in_features=2, out_features=16, bias=True)
      (1): ReLU()
      (2): Linear(in_features=16, out_features=32, bias=True)
      (3): ReLU()
      (4): Linear(in_features=32, out_features=32, bias=True)
    )
  )
  (in_proj): Linear(in_features=1, out_features=16, bias=False)
  (mamba_blocks): ModuleList(
    (0-7): 8 x ResidualBlock(
      (mamba): MambaBlock(
        (in_proj): Linear(in_features=16, out_features=64, bias=False)
        (ssm): S5(
          (ssm): S5SSM()
        )
        (out_proj): Linear(in_features=32, out_features=16, bias=False)
      )
      (norm): RMSNorm((16,), eps=1e-05, elementwise_affine=True)
    )
  )
  (out_proj): Linear(in_features=16, out_features=1, bias=False)
)


In [ ]:
# Train loop
proc_len = h_params.proc_len
init_len = h_params.overlap

writer = SummaryWriter()
if h_params.retrain:
    model.load_state_dict(torch.load("results/S6_model.pth", weights_only=True))
    optimizer.load_state_dict(torch.load("results/S6_optimizer.pth", weights_only=True))

best_loss = float('inf')
for epoch in tqdm(range(h_params.epochs)):
    model.train()
    train_loss = 0
    val_loss = 0
    counter = 0

    #Train in mini-batches
    for data in train_dataloader:
        input_batch = data["input"]
        target_batch = data["target"]
        c = data["c"].to(device)
        # input_batch (batch_size, sequence_length, 1), target_batch (batch_size, sequence_length, 1)
        
        # Determine the number of chunks
        length = (input_batch.shape[1] - init_len) // proc_len

        # Reset model state
        model.reset_state()

        # let the hidden state "warm-up" for init_len samples
        with torch.no_grad():
            model(input_batch[:,:init_len,:].to(device), c)

        # Apply TBPTT
        for j in range(length):
            optimizer.zero_grad()

            # Extract chunks of the input and target signals
            input = input_batch[:,init_len+proc_len*j:init_len+proc_len*(j+1),:].to(device)
            target = target_batch[:,init_len+proc_len*j:init_len+proc_len*(j+1), :].to(device)

            # Forward pass
            output = model(input, c)

            # Compute loss
            loss = loss_fn.compute(output, target)
            #loss = ESR(output, target)

            # Backward pass and optimization
            loss.backward()
            optimizer.step()

            # log loss
            train_loss += loss.item()

        counter += 1

        # Validation
        if counter % 33 == 0:
            with torch.no_grad():
                model.eval()

                for data in val_dataloader:
                    input_batch = data["input"]
                    target_batch = data["target"]
                    c = data["c"].to(device)
                    
                    length = (input_batch.shape[1] - init_len) // proc_len

                    model.reset_state()

                    model(input_batch[:,0:init_len,:].to(device), c)

                    for j in range(length):
                        output = model(input_batch[:,init_len+proc_len*j:init_len+proc_len*(j+1),:].to(device), c)
                        target = target_batch[:,init_len+proc_len*j:init_len+proc_len*(j+1),:].to(device)

                        #loss = F.mse_loss(output, target)
                        loss = loss_fn.compute(output, target)
                        
                        val_loss += loss.item()
            if val_loss < best_loss:
                torch.save(model.state_dict(), "results/S6_model_" + h_params.name + ".pth")
                torch.save(optimizer.state_dict(), "results/S6_optimizer_" + h_params.name + ".pth")
                best_loss = val_loss
            writer.add_scalar('Loss/val', val_loss/len(val_dataloader), epoch+1)

            counter = 0
        
    writer.add_scalar('Loss/train', train_loss/len(train_dataloader), epoch+1)

torch.save(model.state_dict(), "results/final_S6_model_" + h_params.name + ".pth")
torch.save(optimizer.state_dict(), "results/final_S6_optimizer_" + h_params.name + ".pth")
writer.flush()

In [ ]:
# Evaluate (also in 44100 kHz)
import time
import math
proc_len = h_params.proc_len
device = "cpu"

def ESR(output, target):
    tmp = torch.pow(torch.abs(output - target), 2)
    tmp2 = torch.pow(torch.abs(output), 2)
    return torch.sum(tmp) / torch.sum(tmp2)

model = Mamba(args)
model.load_state_dict(torch.load("results/S6_model_" + h_params.name + ".pth", weights_only=True))
fs_orig = 48000
fs = 44100
transform = torchaudio.transforms.Resample(fs_orig, fs)

model.eval()
with torch.no_grad():
    output1 = []
    output2 = []
    target1 = []
    target2 = []
    length = 0

    start_time = time.time()
    for i in range(len(test_data)):
        input_batch = test_data[i]["input"]
        target_batch = test_data[i]["target"]
        c = test_data[i]["c"]

        length += input_batch.shape[0]

        y = model(input_batch.unsqueeze(0), c.unsqueeze(0)).squeeze(0)

        output1.append(y.squeeze(-1))
        target1.append(target_batch.squeeze(-1))

        target_batch = transform(target_batch.T).T
        y = transform(y.T).T

        output2.append(y.squeeze(-1))
        target2.append(target_batch.squeeze(-1))
    print("--- %s seconds/output ---" % ((time.time() - start_time)/length))

    output1 = torch.cat(output1, dim=0)
    output2 = torch.cat(output2, dim=0)
    target1 = torch.cat(target1, dim=0)
    target2 = torch.cat(target2, dim=0)

    esr_value1 = ESR(output1, target1).item()
    esr_value2 = ESR(output2, target2).item()
    print("esr_value = {}".format(esr_value1))
    print("esr_dB_value = {}".format(10*math.log10(abs(esr_value1))))
    print()
    print("esr_value = {}".format(esr_value2))
    print("esr_dB_value = {}".format(10*math.log10(abs(esr_value2))))    
    #torchaudio.save("results/S5_output_48000_" + h_params.name + ".wav", output1.unsqueeze(0), fs_orig)
    #torchaudio.save("results/S5_output_44100_" + h_params.name + ".wav", output2.unsqueeze(0), fs)


--- 8.668184733878303e-06 seconds/output ---
esr_value = 0.00044056258047930896
esr_dB_value = -33.559923927537525

esr_value = 0.00043877633288502693
esr_dB_value = -33.577568058344184


In [7]:
# Function for converting model weights to a JSON file
import torch
import torch.nn as nn
import json
import numpy as np
from models import Mamba

def parse_linear_layers(model, file_dict):
    """
    Parse a PyTorch model and store it in a JSON file.
    Weights from Linear layers and B, C matrices from MambaBlock are transposed before being stored.
    Complex matrices B and C are split into real and imaginary components.
   
    Args:
        model (nn.Module)
        file_dict (dict)
    """
    if 'layers' not in file_dict:
        file_dict['layers'] = []
   
    last_output_features = None
   
    def get_linear_layer_info(layer):
        weights = layer.weight.detach().cpu().numpy()
       
        bias = None
        if layer.bias is not None:
            bias = layer.bias.detach().cpu().numpy().tolist()
       
        layer_info = {
            'type': 'dense',
            'activation': '',
            'shape': [None, layer.out_features],
            'weights': [weights.tolist()]
        }
       
        if bias is not None:
            layer_info['weights'].append(bias)
       
        return layer_info, layer.out_features
    
    def get_linear_params(layer):
        weights = layer.weight.detach().cpu().numpy()
        
        params = {
            'weights': weights.tolist(),
            'bias': None,
            'shape': [None, layer.out_features]
        }
        
        if layer.bias is not None:
            params['bias'] = layer.bias.detach().cpu().numpy().tolist()
            
        return params
    
    def get_norm_params(norm_layer):
        params = {
            'weight': norm_layer.weight.detach().cpu().numpy().tolist(),
            'eps': norm_layer.eps
        }
        return params
   
    def get_mamba_parameters(module):
        # Find S5SSM module which contains the Mamba parameters
        ssm_module = None
        for m in module.modules():
            if isinstance(m, type(module.ssm.ssm)):  # Match the exact type of S5SSM
                ssm_module = m
                break
       
        if ssm_module is None:
            return None
       
        # Extract and process parameters
        A_real = ssm_module.A_real_log.detach().cpu().numpy()
        A_imag = ssm_module.A_imag.detach().cpu().numpy()
        
        # Handle complex B matrix
        B_complex = ssm_module.B.detach().cpu().numpy()
        B_real = np.real(B_complex)
        B_imag = np.imag(B_complex)
        # Transpose both components
        B_real = np.transpose(B_real)
        B_imag = np.transpose(B_imag)
        
        # Handle complex C matrix
        C_complex = ssm_module.C.detach().cpu().numpy()
        C_real = np.real(C_complex)
        C_imag = np.imag(C_complex)
        # Transpose both components
        C_real = np.transpose(C_real)
        C_imag = np.transpose(C_imag)
        
        D = ssm_module.D.detach().cpu().numpy()
        inv_dt = ssm_module.inv_dt.detach().cpu().numpy()
        
        # Get linear layer parameters
        in_proj_params = get_linear_params(module.in_proj) if hasattr(module, 'in_proj') else None
        out_proj_params = get_linear_params(module.out_proj) if hasattr(module, 'out_proj') else None
       
        return {
            'A_real': A_real.tolist(),
            'A_imag': A_imag.tolist(),
            'B_real': B_real.tolist(),
            'B_imag': B_imag.tolist(),
            'C_real': C_real.tolist(),
            'C_imag': C_imag.tolist(),
            'D': D.tolist(),
            'inv_dt': inv_dt.tolist(),
            'in_proj': in_proj_params,
            'out_proj': out_proj_params
        }
    
    def get_residual_block_params(module):
        # Get MambaBlock parameters
        mamba_params = get_mamba_parameters(module.mamba)
        
        # Get RMSNorm parameters
        norm_params = get_norm_params(module.norm)
        
        return {
            'type': 'residual',
            'parameters': {
                'mamba': mamba_params,
                'norm': norm_params
            }
        }
   
    def traverse_model(module):
        nonlocal last_output_features
       
        for name, child in module.named_children():
            if isinstance(child, nn.Linear):
                layer_info, output_features = get_linear_layer_info(child)
                file_dict['layers'].append(layer_info)
                last_output_features = output_features
            elif isinstance(child, nn.ModuleList):
                # Handle ModuleList of ResidualBlocks
                for residual_block in child:
                    residual_params = get_residual_block_params(residual_block)
                    file_dict['layers'].append(residual_params)
                    if hasattr(residual_block.mamba, 'out_proj'):
                        last_output_features = residual_block.mamba.out_proj.out_features
            else:
                traverse_model(child)
   
    traverse_model(model)
    return file_dict

In [8]:
# Import model weights for a real-time plugin
from config import Hyperparams, ModelParams

args = ModelParams()
h_params = Hyperparams()

model = Mamba(args)
model.load_state_dict(torch.load("results/S6_model_" + h_params.name + ".pth", weights_only=True))
print(model)

file_dict = {'in_shape': [None, 1]}
file_dict = parse_linear_layers(model, file_dict)

with open("model_weights.json", "w") as json_file: 
     json.dump(file_dict, json_file, indent=4)

Mamba(
  (film_gen): FiLMGenerator(
    (fc): Sequential(
      (0): Linear(in_features=2, out_features=16, bias=True)
      (1): ReLU()
      (2): Linear(in_features=16, out_features=32, bias=True)
      (3): ReLU()
      (4): Linear(in_features=32, out_features=32, bias=True)
    )
  )
  (in_proj): Linear(in_features=1, out_features=16, bias=False)
  (mamba_blocks): ModuleList(
    (0-7): 8 x ResidualBlock(
      (mamba): MambaBlock(
        (in_proj): Linear(in_features=16, out_features=64, bias=False)
        (ssm): S5(
          (ssm): S5SSM()
        )
        (out_proj): Linear(in_features=32, out_features=16, bias=False)
      )
      (norm): RMSNorm((16,), eps=1e-05, elementwise_affine=True)
    )
  )
  (out_proj): Linear(in_features=16, out_features=1, bias=False)
)
